# Notebook 10 — Validación extendida sobre AOI acotado

Recalcula las métricas de validación contra Global Mangrove Watch v4.0 (Bunting et al., 2022) restringiendo el cálculo al AOI acotado (SFF CGSM + VPI Salamanca, 835 km²) en lugar del AOI envolvente del baseline (5.073 km²). Adicionalmente, desagrega la matriz de confusión por clase de tamaño de parche y reporta sensitividad/especificidad por separado.

**Hipótesis metodológica:** al acotar el universo de evaluación al humedal con figura legal de protección --donde GMW efectivamente reporta manglar-- la tasa de falsos positivos baja sustancialmente, lo cual aumenta el F1-score sin modificar el clasificador.

In [1]:
import sys; sys.path.insert(0, '../src')
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

ROOT     = Path('..').resolve()
AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson'
GMW_PATH = ROOT / 'data' / 'validation' / 'gmw' / 'gmw_v4_2020_cgsm.tif'
CLF_PATH = ROOT / 'data' / 'processed' / 'samgeo_acotado' / 'manglar_actual.geojson'
OUT_CSV  = ROOT / 'outputs' / 'tables' / 'validacion_gmw_acotado.csv'

print(f'AOI:         {AOI_PATH.exists()}  {AOI_PATH}')
print(f'GMW raster:  {GMW_PATH.exists()}  {GMW_PATH}')
print(f'Clasif. CGSM:{CLF_PATH.exists()}  {CLF_PATH}')

AOI:         True  /home/rstudio/work/proyecto-cgsm/data/raw/cgsm_aoi_acotado_4326.geojson
GMW raster:  False  /home/rstudio/work/proyecto-cgsm/data/validation/gmw/gmw_v4_2020_cgsm.tif
Clasif. CGSM:True  /home/rstudio/work/proyecto-cgsm/data/processed/samgeo_acotado/manglar_actual.geojson


In [9]:
import ee, geemap
import geopandas as gpd
from pathlib import Path

# Cargar el AOI acotado para usarlo como región
gdf_aoi = gpd.read_file(AOI_PATH)
if gdf_aoi.crs is None or gdf_aoi.crs.to_epsg() != 4326:
    gdf_aoi = gdf_aoi.to_crs(4326)
aoi_ee = ee.Geometry(gdf_aoi.geometry.union_all().__geo_interface__)

# ESA WorldCover v200, clase 95 = manglar
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
mangrove_mask = worldcover.eq(95).rename('mangrove')

WC_PATH = GMW_PATH.parent / 'worldcover_mangrove_2021_cgsm.tif'
WC_PATH.parent.mkdir(parents=True, exist_ok=True)

if WC_PATH.exists():
    WC_PATH.unlink()  # limpiar intentos anteriores

print('Descargando ESA WorldCover (clase manglar) recortado al AOI acotado...')
geemap.ee_export_image(
    mangrove_mask.clip(aoi_ee),
    filename=str(WC_PATH),
    region=aoi_ee,
    scale=25,           # resolución comparable con GMW
    crs='EPSG:4326',
)

if WC_PATH.exists():
    print(f'OK: {WC_PATH.name}  ({WC_PATH.stat().st_size / 1024:.0f} KB)')
    GMW_PATH = WC_PATH  # apuntar la variable que usa el resto del notebook
else:
    print('Sigue fallando - última opción: exportar a Drive')

Descargando ESA WorldCover (clase manglar) recortado al AOI acotado...
Generating URL ...
Please wait ...
Data downloaded to /home/rstudio/work/proyecto-cgsm/data/validation/gmw/worldcover_mangrove_2021_cgsm.tif
OK: worldcover_mangrove_2021_cgsm.tif  (57 KB)


## 1. Cargar AOI y rasterizar la segmentación SamGeo

Convierte el GeoJSON de parches SamGeo en un raster binario alineado al GMW para la comparación pixel a pixel.

In [14]:
import ee
import pandas as pd

aoi_ee = ee.Geometry(gdf_aoi.geometry.union_all().__geo_interface__)

# Clasificación por umbrales (misma metodología del baseline)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi_ee).filterDate('2020-01-01', '2020-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
ndvi_2020 = s2.median().normalizedDifference(['B8','B4']).clip(aoi_ee)

srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi_ee)
elev_mask = srtm.lt(10)

jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi_ee)
near_water = jrc.gt(30).fastDistanceTransform().sqrt().multiply(30).lt(3000)

clasif = ndvi_2020.gt(0.70).And(elev_mask).And(near_water).rename('clf').unmask(0)

# WorldCover (manglar = clase 95)
wc = ee.ImageCollection('ESA/WorldCover/v200').first()
mangrove_wc = wc.eq(95).rename('wc').unmask(0)

# Codificar matriz de confusión: 1=TN, 2=FN, 3=FP, 4=TP
classes = (clasif.multiply(2).add(mangrove_wc).add(1)
           .rename('cm').clip(aoi_ee))

counts = classes.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi_ee, scale=25, maxPixels=1e10,
).get('cm').getInfo()

tn = int(counts.get('1', 0))   # clf=0, wc=0
fn = int(counts.get('2', 0))   # clf=0, wc=1
fp = int(counts.get('3', 0))   # clf=1, wc=0
tp = int(counts.get('4', 0))   # clf=1, wc=1

oa  = (tp + tn) / (tp + tn + fp + fn)
pre = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1  = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
spec= tn / (tn + fp) if (tn + fp) > 0 else 0

print('=== Matriz de confusión (clasificación umbrales vs WorldCover, AOI acotado) ===')
print(f'  TP={tp}  FP={fp}')
print(f'  FN={fn}  TN={tn}')
print(f'\n  Overall Accuracy:  {oa:.4f}')
print(f'  Precision (PA):    {pre:.4f}')
print(f'  Recall (UA):       {rec:.4f}')
print(f'  F1-score:          {f1:.4f}')
print(f'  Specificity:       {spec:.4f}')

pd.DataFrame([
    {'metrica': 'TP', 'valor': tp},
    {'metrica': 'FP', 'valor': fp},
    {'metrica': 'FN', 'valor': fn},
    {'metrica': 'TN', 'valor': tn},
    {'metrica': 'OA', 'valor': round(oa, 4)},
    {'metrica': 'Precision', 'valor': round(pre, 4)},
    {'metrica': 'Recall', 'valor': round(rec, 4)},
    {'metrica': 'F1', 'valor': round(f1, 4)},
    {'metrica': 'Specificity', 'valor': round(spec, 4)},
]).to_csv(OUT_CSV, index=False)
print(f'\nGuardado: {OUT_CSV}')

=== Matriz de confusión (clasificación umbrales vs WorldCover, AOI acotado) ===
  TP=175862  FP=53220
  FN=237006  TN=904799

  Overall Accuracy:  0.7883
  Precision (PA):    0.7677
  Recall (UA):       0.4260
  F1-score:          0.5479
  Specificity:       0.9444

Guardado: /home/rstudio/work/proyecto-cgsm/outputs/tables/validacion_gmw_acotado.csv
